In [1]:
# this notebook will prepare data to visualize on chart
# create markers
import json
import pandas as pd
import gc
from pathlib import Path
import numpy as np

DATA_PATH = Path.cwd().parents[1] / "data" / "bigData" / "BTCUSD-15m.json"

with open(DATA_PATH) as f:
    file = json.load(f)

ohlcv = file["ohlcv"]  # list of [timestamp, open, high, low, close, volume]
del file  # free the rest of the JSON (symbol, onchart, etc.), , preserve RAM

df = pd.DataFrame(ohlcv, columns=["timestamp", "open", "high", "low", "close", "vol"])
del ohlcv  # free the Python list-of-lists, preserve RAM
gc.collect()

# Halve memory: float64 -> float32 for price/volume columns
df[["open", "high", "low", "close", "vol"]] = df[["open", "high", "low", "close", "vol"]].astype("float32")

# print(df.head())
# beware  was truncated by the NotebookEdit tool on the previous write
print(f"Shape: {df.shape} | Memory: {df.memory_usage(deep=True).sum()/ 1e6:.2f} MB")

Shape: (70930, 6) | Memory: 1.99 MB


In [2]:
df.head()
# save as JSONL
# OUT_PATH = Path.cwd().parents[1] / "data" / "bigData" / "BTCUSD-15m.jsonl"
# df.to_json(OUT_PATH, orient="records", lines=True)
# print(f"Saved {len(df)} rows to {OUT_PATH}")

,timestamp,open,high,low,close,vol
0,1708659900000,51099.929688,51120.000000,51052.199219,51100.000000,209.838379
1,1708660800000,51100.000000,51106.871094,51003.000000,51091.859375,230.937881
2,1708661700000,51091.871094,51236.640625,51031.101562,51236.640625,333.208618
3,1708662600000,51236.628906,51321.921875,51183.210938,51224.031250,284.106201
4,1708663500000,51224.039062,51254.199219,51195.839844,51219.589844,169.352463


# What are we doing here
- Create ICT pivot point, STR, ITR and LTR

## Additional features since ICT pivot point was weak
- Volume - use that raw vol -> standardize later
- Time of day -> convert to int? 0000-2345 
- Momentum / acceleration - use Schaff Trend Cycle 0-100
- Volatility regime - ATR14

## exclude - don't know what to pick
- HTF trend direction (broader market context beyond ITR/LTR)
    - i.e. Trend, Support/Resistance level. How the f do I calculate support/resistance of interest except keylv ?

In [3]:
def _fix_break_of_structure(isXXX, isSource, high, low):
    """
    Ensure isXXX alternates strictly between 1 and -1.
    On a break (1,1 or -1,-1), insert the best counter swing between the two offending positions,
    but ONLY if it is structurally valid:
      - Inserting a  1 (high): candidate high must be > max(low[p1],  low[p2])
      - Inserting a -1 (low) : candidate low  must be < min(high[p1], high[p2])
    If no valid candidate exists, remove the less extreme of the two duplicates instead.
    """
    changed = True
    while changed:
        changed = False
        non_zero = np.where(isXXX != 0)[0]

        for i in range(1, len(non_zero)):
            p1, p2 = non_zero[i - 1], non_zero[i]
            v1, v2 = isXXX[p1], isXXX[p2]

            if v1 != v2:
                continue

            counter    = np.int8(-v1)
            candidates = np.arange(p1 + 1, p2)

            inserted = False
            if len(candidates) > 0:
                source_cands = candidates[isSource[candidates] == counter]
                pool = source_cands if len(source_cands) > 0 else candidates

                if counter == -1:
                    # Inserting a low: must dip below both surrounding highs
                    threshold  = min(high[p1], high[p2])
                    valid_pool = pool[low[pool] < threshold]
                else:
                    # Inserting a high: must rise above both surrounding lows
                    threshold  = max(low[p1], low[p2])
                    valid_pool = pool[high[pool] > threshold]

                if len(valid_pool) > 0:
                    best = (valid_pool[np.argmin(low[valid_pool])]
                            if counter == -1
                            else valid_pool[np.argmax(high[valid_pool])])
                    isXXX[best] = counter
                    inserted = True

            if not inserted:
                # No structurally valid candidate — drop the less extreme duplicate
                if v1 == 1:   # two highs: keep the higher one
                    isXXX[p1 if high[p1] < high[p2] else p2] = 0
                else:          # two lows: keep the lower one
                    isXXX[p1 if low[p1]  > low[p2] else p2] = 0

            changed = True
            break  # restart after each change

    return isXXX


In [4]:
def _fix_price_order(isXXX, high, low):
    """
    Fix price ordering violations by inserting 2 bridging marks instead of removing.
    For ITR/LTR: inserted marks need not be original swing points.
    Falls back to removal only if no valid bridge candidates exist.
    """
    changed = True
    while changed:
        changed = False
        non_zero = np.where(isXXX != 0)[0]

        for i in range(1, len(non_zero)):
            p1, p2 = non_zero[i-1], non_zero[i]
            v1, v2 = isXXX[p1], isXXX[p2]

            if v1 == 1 and v2 == -1 and low[p2] >= high[p1]:
                # Need a real low then a real high between p1 and p2
                cands = np.arange(p1 + 1, p2)
                valid_lows = cands[low[cands] < high[p1]]
                if len(valid_lows) > 0:
                    best_low = valid_lows[np.argmin(low[valid_lows])]
                    after = np.arange(best_low + 1, p2)
                    valid_highs = after[high[after] > low[best_low]]
                    if len(valid_highs) > 0:
                        best_high = valid_highs[np.argmax(high[valid_highs])]
                        isXXX[best_low]  = -1
                        isXXX[best_high] = 1
                        changed = True
                        break
                # fallback: remove the bad low
                isXXX[p2] = 0
                changed = True
                break

            elif v1 == -1 and v2 == 1 and high[p2] <= low[p1]:
                # Need a real high then a real low between p1 and p2
                cands = np.arange(p1 + 1, p2)
                valid_highs = cands[high[cands] > low[p1]]
                if len(valid_highs) > 0:
                    best_high = valid_highs[np.argmax(high[valid_highs])]
                    after = np.arange(best_high + 1, p2)
                    valid_lows = after[low[after] < high[best_high]]
                    if len(valid_lows) > 0:
                        best_low = valid_lows[np.argmin(low[valid_lows])]
                        isXXX[best_high] = 1
                        isXXX[best_low]  = -1
                        changed = True
                        break
                # fallback: remove the bad high
                isXXX[p2] = 0
                changed = True
                break

    return isXXX


In [5]:
# varsion 1
def strHunt(df, window=4):
    n = len(df)
    df = df.copy()

    high = df['high'].values
    low  = df['low'].values

    isSTR = np.zeros(n, dtype=np.int8)

    # --- First window: can't look backward, so determine direction from the span ---
    # if n > window + 1:
    #     diff = high[window + 1] - low[0]
    #     if diff > 0:
    #         isSTR[0] = -1   # start of bullish move → mark as low
    #     elif diff < 0:
    #         isSTR[0] = 1    # start of bearish move → mark as high

    # --- First window: method 2 just mark 0

    # --- Main loop: candles with a full symmetric window on both sides ---
    # Stop at n-window so the last `window` candles remain 0
    for i in range(window, n - window):
        win_high = high[i - window : i + window + 1]
        win_low  = low [i - window : i + window + 1]

        is_swing_high = high[i] >= win_high.max()   # local maximum
        is_swing_low  = low[i]  <= win_low.min()    # local minimum

        if is_swing_high and not is_swing_low:
            isSTR[i] = 1    # confirmed swing high
        elif is_swing_low and not is_swing_high:
            isSTR[i] = -1   # confirmed swing low
        # neither → stays 0
        # both, liquidity swept on both high and low → stays 0
        

    isSTR = _fix_break_of_structure(isSTR, isSTR, high, low)
    isSTR = _fix_price_order(isSTR, high, low)

    df['isSTR'] = isSTR
    return df

# execute -- 
df = strHunt(df)

In [6]:
def itrHunt(df, group=3):
    # there must be isSTR column presented first
    # search for intermediate term range
    # group params input must be odd number (has middle point)
    # to qualify, the middle swing high must be the highest among `group` consecutive swing HIGHS
    # to qualify, the middle swing low must be the lowest among `group` consecutive swing LOWS
    # highs are compared against highs only, lows against lows only
    # (requires at least 1,1,1 or -1,-1,-1 in isSTR for a valid group comparison)

    df = df.copy()
    n = len(df)

    high  = df['high'].values
    low   = df['low'].values
    isSTR = df['isSTR'].values
    isITR = np.zeros(n, dtype=np.int8)

    # Separate by type — compare like with like
    high_pos = np.where(isSTR == 1)[0]   # all swing high positions
    low_pos  = np.where(isSTR == -1)[0]  # all swing low positions

    half = group // 2  # e.g. group=3 → half=1

    # ITR swing highs: middle must be highest among `group` consecutive swing highs
    for j in range(half, len(high_pos) - half):
        group_pos = high_pos[j - half : j + half + 1]
        mid_pos   = high_pos[j]
        if high[mid_pos] >= high[group_pos].max():
            isITR[mid_pos] = 1

    # ITR swing lows: middle must be lowest among `group` consecutive swing lows
    for j in range(half, len(low_pos) - half):
        group_pos = low_pos[j - half : j + half + 1]
        mid_pos   = low_pos[j]
        if low[mid_pos] <= low[group_pos].min():
            isITR[mid_pos] = -1

    # handle break of structure case
    isITR = _fix_break_of_structure(isITR, isSTR, high, low)
    # handle exception case : price ordering
    isITR = _fix_price_order(isITR, high, low)


    df['isITR'] = isITR
    return df


# execute --
df = itrHunt(df)

In [7]:
def ltrHunt(df, group=3):
    # there must be isITR column presented first
    # search for long term range
    # highs are compared against highs only, lows against lows only
    # (requires at least 1,1,1 or -1,-1,-1 in isITR for a valid group comparison)

    df = df.copy()
    n = len(df)

    high  = df['high'].values
    low   = df['low'].values
    isITR = df['isITR'].values
    isLTR = np.zeros(n, dtype=np.int8)

    # Separate by type — compare like with like
    high_pos = np.where(isITR == 1)[0]
    low_pos  = np.where(isITR == -1)[0]

    half = group // 2

    for j in range(half, len(high_pos) - half):
        group_pos = high_pos[j - half : j + half + 1]
        mid_pos   = high_pos[j]
        if high[mid_pos] >= high[group_pos].max():
            isLTR[mid_pos] = 1

    for j in range(half, len(low_pos) - half):
        group_pos = low_pos[j - half : j + half + 1]
        mid_pos   = low_pos[j]
        if low[mid_pos] <= low[group_pos].min():
            isLTR[mid_pos] = -1

    # handle break of structure case
    isLTR = _fix_break_of_structure(isLTR, isITR, high, low)
    # handle exception case : price ordering
    isLTR = _fix_price_order(isLTR, high, low)


    df['isLTR'] = isLTR
    return df


# execute --
df = ltrHunt(df)

In [8]:
def validate_markers(df, col):
    """
    Validate a marker column (e.g. 'isITR', 'isLTR') across the entire DataFrame.

    Checks two invariants:
      1. BOS (Break-of-Structure): consecutive non-zero entries must alternate sign (no 1,1 or -1,-1).
      2. Price order: after a high (1), the next low (-1) must be strictly below it,
         and after a low (-1), the next high (1) must be strictly above it.

    Returns
    -------
    dict with keys:
        'bos_violations'   : DataFrame of rows where sign repeats
        'order_violations' : DataFrame of rows where price order is broken
        'is_valid'         : True only if both checks pass
    """
    rows = df[df[col] != 0][['timestamp', 'high', 'low', col]].copy()
    rows['price_at_mark'] = rows.apply(
        lambda r: r['high'] if r[col] == 1 else r['low'], axis=1
    )

    vals = rows[col].values

    # --- BOS check: consecutive same-sign entries ---
    bos_mask = [False] + [vals[i] == vals[i - 1] for i in range(1, len(vals))]
    bos_violations = rows[bos_mask]

    # --- Price order check ---
    order_fail_idx = []
    for i in range(1, len(rows)):
        prev = rows.iloc[i - 1]
        curr = rows.iloc[i]
        if curr[col] == 1:          # current is a high → must be above prev low
            if curr['high'] <= prev['low']:
                order_fail_idx.append(rows.index[i])
        else:                        # current is a low → must be below prev high
            if curr['low'] >= prev['high']:
                order_fail_idx.append(rows.index[i])
    order_violations = rows.loc[order_fail_idx]

    # --- Print summary ---
    label = col
    total = len(rows)
    print(f"=== {label} validation ({total} marks across full dataset) ===")

    if len(bos_violations):
        print(f"  BOS violations : {len(bos_violations)}")
        print(bos_violations[['timestamp', 'high', 'low', col, 'price_at_mark']].to_string())
    else:
        print("  BOS violations : 0  — alternation OK")

    if len(order_violations):
        print(f"  Order violations: {len(order_violations)}")
        print(order_violations[['timestamp', 'high', 'low', col, 'price_at_mark']].to_string())
    else:
        print("  Order violations: 0  — price ordering OK")

    is_valid = len(bos_violations) == 0 and len(order_violations) == 0
    print(f"  Valid: {is_valid}\n")

    return {
        'bos_violations':   bos_violations,
        'order_violations': order_violations,
        'is_valid':         is_valid,
    }


In [9]:
# --- run validation on STR columns ---
str_report = validate_markers(df, 'isSTR')


=== isSTR validation (12644 marks across full dataset) ===
  BOS violations : 0  — alternation OK
  Order violations: 0  — price ordering OK
  Valid: True



In [10]:
# --- run validation on ITR columns ---
itr_report = validate_markers(df, 'isITR')

=== isITR validation (3992 marks across full dataset) ===
  BOS violations : 0  — alternation OK
  Order violations: 0  — price ordering OK
  Valid: True



In [11]:
# --- run validation on LTR columns ---
ltr_report = validate_markers(df, 'isLTR')

=== isLTR validation (1277 marks across full dataset) ===
  BOS violations : 0  — alternation OK
  Order violations: 0  — price ordering OK
  Valid: True



In [12]:
# check point - preview data
df.tail()

,timestamp,open,high,low,close,vol,isSTR,isITR,isLTR
70925,1772492400000,69440.476562,69479.578125,69215.398438,69223.242188,121.096069,0,0,0
70926,1772493300000,69223.250000,69261.687500,68990.976562,69002.929688,223.747665,0,0,0
70927,1772494200000,69002.921875,69014.257812,68758.398438,68796.859375,195.868637,0,0,0
70928,1772495100000,68796.859375,68965.671875,68740.000000,68830.062500,97.017410,0,0,0
70929,1772496000000,68830.062500,69004.398438,68750.679688,68809.992188,168.970734,0,0,0


# create time feature
```
**Time of day**
`time_sin`, `time_cos` (sin/cos of minutes-since-midnight, period=1440). 96 values for 15m bars — circularity matters (23:45 → 00:00 gap).
**Day of week** one-hot `[is_mon, is_tue, is_wed, is_thu, is_fri, is_sat, is_sun]`. Captures weekend staleness. No sin/cos needed at 7 values; avoids false ordinality of raw 0–6 int. **7 features.**
```

In [13]:
# --- Time features ---
dt = pd.to_datetime(df["timestamp"], unit="ms", utc=True)

# Time of day: sin/cos of minutes-since-midnight (period = 1440)
minutes = dt.dt.hour * 60 + dt.dt.minute
df["time_sin"] = np.sin(2 * np.pi * minutes / 1440).astype("float32")
df["time_cos"] = np.cos(2 * np.pi * minutes / 1440).astype("float32")

# Day of week: one-hot (Monday=0 … Sunday=6)
dow = dt.dt.dayofweek  # 0=Mon, 6=Sun
for i, name in enumerate(["is_mon", "is_tue", "is_wed", "is_thu", "is_fri", "is_sat", "is_sun"]):
    df[name] = (dow == i).astype("int8")


In [14]:
# check point
df.tail()

,timestamp,open,high,low,close,vol,isSTR,isITR,isLTR,time_sin,time_cos,is_mon,is_tue,is_wed,is_thu,is_fri,is_sat,is_sun
70925,1772492400000,69440.476562,69479.578125,69215.398438,69223.242188,121.096069,0,0,0,-0.258819,0.965926,1,0,0,0,0,0,0
70926,1772493300000,69223.250000,69261.687500,68990.976562,69002.929688,223.747665,0,0,0,-0.195090,0.980785,1,0,0,0,0,0,0
70927,1772494200000,69002.921875,69014.257812,68758.398438,68796.859375,195.868637,0,0,0,-0.130526,0.991445,1,0,0,0,0,0,0
70928,1772495100000,68796.859375,68965.671875,68740.000000,68830.062500,97.017410,0,0,0,-0.065403,0.997859,1,0,0,0,0,0,0
70929,1772496000000,68830.062500,69004.398438,68750.679688,68809.992188,168.970734,0,0,0,0.000000,1.000000,0,1,0,0,0,0,0


# Schaff Trend Cycle
```markdown
MACD = EMA(close,23)-EMA(close,50)
%KMACD = MACD-minMACD/ (maxMACD-minMACD) * 100 : min, max lookback in 10 period window
PF = EMA(%KMACD,3)
%KPF = PF-minPF/ (maxPF-minPF) * 100 : min, max lookback in 10 period window
STC = EMA(%KPF,3)
```

```
EMA_t = (price_t * a) + (EMA_(t-1)*(1-a)) where
a = 2/ (period + 1)
```


In [15]:
def ema(series, period):
    """Exponential Moving Average — matches TradingView/most platform behavior."""
    k = 2 / (period + 1)
    result = np.empty(len(series), dtype="float32")
    result[:] = np.nan

    # seed: first non-NaN value
    start = np.argmax(~np.isnan(series))
    result[start] = series[start]

    for i in range(start + 1, len(series)):
        result[i] = series[i] * k + result[i - 1] * (1 - k)

    return result

def percent_k(series, period=10):
    """Stochastic %K over a rolling window — used in Schaff Trend Cycle."""
    result = np.full(len(series), np.nan, dtype="float32")

    for i in range(period - 1, len(series)):
        window = series[i - period + 1 : i + 1]
        lo, hi = np.nanmin(window), np.nanmax(window)
        denom = hi - lo
        result[i] = 0.0 if denom == 0 else (series[i] - lo) / denom * 100

    return result

def stc(close, fast=23, slow=50, k_period=10, smooth=3):
    """Schaff Trend Cycle (0–100)."""
    macd     = ema(close, fast) - ema(close, slow)
    pct_macd = percent_k(macd, k_period)
    pf       = ema(pct_macd, smooth)
    pct_pf   = percent_k(pf, k_period)
    return ema(pct_pf, smooth)

# execute
df["stc_0510"] = stc(df["close"].values, fast=5, slow=10)

In [16]:
# check point 
df.tail()

,timestamp,open,high,low,close,vol,isSTR,isITR,isLTR,time_sin,time_cos,is_mon,is_tue,is_wed,is_thu,is_fri,is_sat,is_sun,stc_0510
70925,1772492400000,69440.476562,69479.578125,69215.398438,69223.242188,121.096069,0,0,0,-0.258819,0.965926,1,0,0,0,0,0,0,31.910965
70926,1772493300000,69223.250000,69261.687500,68990.976562,69002.929688,223.747665,0,0,0,-0.195090,0.980785,1,0,0,0,0,0,0,15.955482
70927,1772494200000,69002.921875,69014.257812,68758.398438,68796.859375,195.868637,0,0,0,-0.130526,0.991445,1,0,0,0,0,0,0,7.977741
70928,1772495100000,68796.859375,68965.671875,68740.000000,68830.062500,97.017410,0,0,0,-0.065403,0.997859,1,0,0,0,0,0,0,3.988871
70929,1772496000000,68830.062500,69004.398438,68750.679688,68809.992188,168.970734,0,0,0,0.000000,1.000000,0,1,0,0,0,0,0,1.994435


# Volatility regime
use ATR14

```
True_range = max[(High - Low), abs(High - Close_prev), abs(Low - Close_prev)]
```

In [17]:
def atr(high, low, close, period=14):
    """ATR using Wilder's RMA (matches TradingView). alpha = 1/period."""
    n = len(close)
    tr = np.empty(n, dtype="float32")
    tr[0] = high[0] - low[0]
    for i in range(1, n):
        tr[i] = max(high[i] - low[i],
                    abs(high[i] - close[i - 1]),
                    abs(low[i]  - close[i - 1]))

    result = np.full(n, np.nan, dtype="float32")
    result[period - 1] = tr[:period].mean()   # seed: simple average of first `period` TRs
    alpha = 1.0 / period
    for i in range(period, n):
        result[i] = tr[i] * alpha + result[i - 1] * (1 - alpha)

    return result


h, l, c = df["high"].values, df["low"].values, df["close"].values

raw = atr(h, l, c, period=14)
df["atr_14_pct"] = (raw / c).astype("float32")

df[["atr_14_pct"]].tail(10)


,atr_14_pct
70920,0.005443
70921,0.005358
70922,0.005215
70923,0.004992
70924,0.004838
70925,0.004780
70926,0.004733
70927,0.004673
70928,0.004572
70929,0.004510


In [18]:
# preview check point
df.tail()

,timestamp,open,high,low,close,vol,isSTR,isITR,isLTR,time_sin,time_cos,is_mon,is_tue,is_wed,is_thu,is_fri,is_sat,is_sun,stc_0510,atr_14_pct
70925,1772492400000,69440.476562,69479.578125,69215.398438,69223.242188,121.096069,0,0,0,-0.258819,0.965926,1,0,0,0,0,0,0,31.910965,0.004780
70926,1772493300000,69223.250000,69261.687500,68990.976562,69002.929688,223.747665,0,0,0,-0.195090,0.980785,1,0,0,0,0,0,0,15.955482,0.004733
70927,1772494200000,69002.921875,69014.257812,68758.398438,68796.859375,195.868637,0,0,0,-0.130526,0.991445,1,0,0,0,0,0,0,7.977741,0.004673
70928,1772495100000,68796.859375,68965.671875,68740.000000,68830.062500,97.017410,0,0,0,-0.065403,0.997859,1,0,0,0,0,0,0,3.988871,0.004572
70929,1772496000000,68830.062500,69004.398438,68750.679688,68809.992188,168.970734,0,0,0,0.000000,1.000000,0,1,0,0,0,0,0,1.994435,0.004510


# Export Data

# Export augmented data to json visualize the data on Chart

# onchart data
```json
onchart : [{
    name: "Marker",
    type: "Marker",
    data: marking,
    settings: {
        "z-index": 5,
        "legend": false
    }
}]
```

where marking is

```json
[
     1771612200000,  // timestamp
     'buy',          // marker type: 'buy' | 'sell' | 'str' | 'itr' | 'ltr'
     67527.01,       // value at mark
     "lo"            // legend (optional)
], ...
```



# off chart data
```json
"offchart": [
        {
            "name": "STC, 5, 10",
            "type": "STC",
            "data":[[
                timestamp, value
            ],...],
            "settings": {
                "upper": 70,
                "lower": 30,
                "backColor": "#9b9ba316",
                "bandColor": "#666"
            }
        },

In [19]:
def exportJsonOverlays(df, out_path):
    itr_df = df[df['isITR'] != 0][['timestamp', 'isITR', 'high', 'low']]

    marker_data = [
        [
            int(row['timestamp']),
            'itr',
            float(row['high']) if row['isITR'] == 1 else float(row['low']),
            ''
        ]
        for _, row in itr_df.iterrows()
    ]

    stc_data  = [[int(ts), float(v)] for ts, v in zip(df['timestamp'], df['stc_0510'])]
    natr_data = [[int(ts), float(v)] for ts, v in zip(df['timestamp'], df['atr_14_pct'])]

    payload = {
        "onchart": [{
            "name": "Marker",
            "type": "Marker",
            "data": marker_data,
            "settings": {
                "z-index": 5,
                "legend": False
            }
        }],
        "offchart": [
            {
                "name": "STC, 5, 10",
                "type": "STC",
                "data": stc_data,
                "settings": {
                    "upper": 70,
                    "lower": 30,
                    "backColor": "#9b9ba316",
                    "bandColor": "#666"
                }
            },
            {
                "name": "%NATR, 14",
                "type": "ATR",
                "data": natr_data,
                "settings": {}
            }
        ]
    }

    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with open(out_path, 'w') as f:
        json.dump(payload, f)

    print(f"Exported {len(marker_data)} markers to {out_path}")


# execute
OUT_PATH = Path.cwd().parents[1] / "data" / "bigData" / "15m-overlay-v2.json"
exportJsonOverlays(df, OUT_PATH)


Exported 3992 markers to /Users/aimliu/Library/CloudStorage/OneDrive-Personal/_CODE/Python/py-CandleScience/data/bigData/15m-overlay-v2.json
